# Curriculum 04 · Lab 6 — Self-query: natural language turns into a metadata filter

**Goal:** Make the retriever understand constraints, not just semantics.
"Who founded Montevideo? **from bucket b1**" — the second half is not a
semantic signal, it is a *filter*. Self-query hands the question to a small
LLM that parses it into a structured query (semantic part + optional metadata
filter), then runs the filter natively in the vector store.

```
Retriever : SelfQueryRetriever (langchain-classic)
Parser    : llama-3.3-70b-versatile (ChatGroq) -> StructuredQuery
Fields    : bucket (b1/b2/b3) + source — the filterable metadata
Translator: ChromaTranslator passed EXPLICITLY (langchain-classic 1.0.8's
            auto-detection crashes on langchain-community 1.x — VERIFIED)
Paths     : (a) pure semantic (no filter) | (b) NL constraint -> filter
            | (c) hand-written filter (ground truth, no LLM)
Embedding : BGE (BAAI/bge-base-en-v1.5, local, CPU)
```

**Why self-query:** metadata filtering (Project 14) is powerful but requires
the caller to know the metadata schema and write filters by hand. Self-query
moves that burden to the LLM: the same question that drives retrieval also
produces the filter, with zero schema awareness in the caller.

This is the last lab of track 04-retrieval (see
`.omo/plans/layer1-rag-playbook.md`).


## 0 · Setup — environment, imports & repo paths

**WHAT:** Installs the lab's dependencies (a no-op if already present),
loads `GROQ_API_KEY` from the repo-root `.env`, and puts the repo root on
`sys.path` so every repo-relative path behaves exactly like the lab script.

**WHY:** Everything embeds **locally** with BGE into an **ephemeral** Chroma
store (in-memory — nothing written to disk). The only API call is the query
parser LLM — Groq's `llama-3.3-70b-versatile` (a commented Gemini alternative
is kept in the source). The imports cell resolves the repo root by walking up
from the kernel cwd and `cd`s into it.

**WHAT TO EXPECT:** no output from the pip cell (packages already
installed), a silent import from the second. The BGE model loads lazily when
the experiment cell first calls it; the Groq key is read from `.env`.


In [1]:
# Lab-specific dependencies (already in requirements.txt — the install is a
# no-op safety net for fresh environments):
#   sentence-transformers -> local BGE embeddings (langchain_huggingface)
#   langchain-chroma      -> the ephemeral (in-memory) Chroma store
#   langchain-classic     -> SelfQueryRetriever + the query-constructor schema
#   langchain-community   -> ChromaTranslator (explicit translator workaround)
#   langchain-groq        -> ChatGroq (the query parser LLM)
#   python-dotenv         -> loads GROQ_API_KEY from the repo-root .env
#   pandas                -> reads the passages/test.parquet corpus
%pip install sentence-transformers langchain-chroma langchain-classic langchain-community langchain-groq python-dotenv pandas



[notice] A new release of pip is available: 24.2 -> 26.2
[notice] To update, run: python -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


In [2]:
from __future__ import annotations

import os
import sys
import time
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv

# Make the repo-root component library importable. A notebook has no
# ``__file__``, so resolve the repo root by walking up from the kernel's
# working directory — this works whether the kernel launches from the repo
# root (like the lab script) or from the notebook's own folder (Jupyter's
# default) — then cd into it so every repo-relative path behaves exactly
# like the .py.
REPO_ROOT = Path.cwd()
for candidate in (Path.cwd(), *Path.cwd().parents):
    if (candidate / "curriculum").is_dir() and (candidate / "NoteBooks").is_dir():
        REPO_ROOT = candidate
        break
os.chdir(REPO_ROOT)
sys.path.insert(0, str(REPO_ROOT))

from langchain_chroma import Chroma  # noqa: E402
from langchain_classic.chains.query_constructor.schema import AttributeInfo  # noqa: E402
from langchain_classic.retrievers import SelfQueryRetriever  # noqa: E402
from langchain_community.query_constructors.chroma import ChromaTranslator  # noqa: E402
from langchain_core.documents import Document  # noqa: E402
from langchain_groq import ChatGroq  # noqa: E402
from langchain_huggingface import HuggingFaceEmbeddings  # noqa: E402
# (Gemini alternative: from langchain_google_genai import ChatGoogleGenerativeAI)


/tmp/ipykernel_1406946/4275630045.py:28: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.query_constructors.chroma import ChromaTranslator  # noqa: E402


## 1 · Configuration — the experiment's knobs

**WHAT:** The corpus constants (`N_PASSAGES = 60`) plus the self-query knobs:
`K = 5` (every search path returns this many docs), `BUCKETS` (the three
synthetic metadata values cycled over passages), `QUERY_PLAIN` (pure semantic
— no filter expected), `QUERY_FILTERED` (same sentence + "from bucket b1" —
a filter must be emitted), and `MAX_PARSE_ATTEMPTS = 2` (retries for the LLM
filter-parse on path (b) only).

**WHY:** The bucket tags are synthetic — attached by index, not by content —
so the gate can prove the filter worked *because of the metadata*, not the
semantics: path (b) must return only bucket-b1 docs, and path (c) (the
hand-written filter) is the no-LLM ground truth it is compared against.


In [3]:
PASSAGES_PATH = Path("Data/corpus/rag-mini-wikipedia/passages.parquet")
N_PASSAGES = 60  # deterministic head of the 3200-passage corpus (keeps runtime low)
K = 5  # top-k for every search path
BUCKETS = ("b1", "b2", "b3")  # synthetic metadata buckets assigned round-robin
SOURCE_NAME = "rag-mini-wikipedia"
QUERY_PLAIN = "Who founded Montevideo?"  # (a) pure semantic — no filter expected
QUERY_FILTERED = "Who founded Montevideo? from bucket b1"  # (b) NL constraint
FILTER_BUCKET = "b1"  # the bucket requested in QUERY_FILTERED / path (c)
PREVIEW = 62  # max characters of passage text shown next to each hit
MAX_PARSE_ATTEMPTS = 2  # retries for the LLM filter-parse on path (b) only
EMBED_MODEL_NAME = "BAAI/bge-base-en-v1.5"
EMBED_DIM = 768
LLM_MODEL = "llama-3.3-70b-versatile"  # Groq query parser, never the embedder
# (Gemini alternative: LLM_MODEL = "gemini-2.5-flash" — needs GOOGLE_API_KEY in .env)
LLM_TEMPERATURE = 0.0


## 2 · Load — corpus passages + synthetic metadata buckets

**WHAT:** `load_passages` pulls the first `n` passages from `passages.parquet`;
`attach_metadata` wraps each into a Document tagged round-robin with a
`bucket` (b1, b2, b3) plus the `source` name; `preview` / `buckets_of` are
the display helpers.

**WHY:** The metadata is the whole subject of this lab — self-query exists
to turn "from bucket b1" into a store-level filter, and the synthetic tags
make the filter's effect unambiguous.


In [4]:
def load_passages(path: Path, n: int) -> list[str]:
    """Return the first ``n`` passage texts (deterministic, no randomness)."""
    df = pd.read_parquet(path)
    return df["passage"].head(n).tolist()


def attach_metadata(texts: list[str]) -> list[Document]:
    """Wrap passages in Documents with synthetic, deterministic metadata."""
    return [
        Document(
            page_content=text,
            metadata={
                # Round-robin buckets: b1, b2, b3, b1, b2, … (every third doc in b1)
                "bucket": BUCKETS[i % len(BUCKETS)],
                "source": SOURCE_NAME,
            },
        )
        for i, text in enumerate(texts)
    ]


def preview(text: str, limit: int = PREVIEW) -> str:
    """Flatten a passage for one-line printing."""
    flat = text.replace("\n", " ")
    return flat[:limit] + ("..." if len(flat) > limit else "")


def buckets_of(docs: list[Document]) -> list[str]:
    """The metadata bucket of every returned doc, in rank order."""
    return [d.metadata.get("bucket", "?") for d in docs]


## 3 · Experiment — embed, index, self-query on three paths

**WHAT:** `run_experiment` embeds the 60 passages with BGE into an ephemeral
Chroma store, builds the `SelfQueryRetriever` (Groq parser +
`ChromaTranslator` passed explicitly — the documented workaround for
langchain-classic 1.0.8), then runs three paths per question: (a) the pure
semantic query, (b) the same sentence with an NL bucket constraint, and (c)
the same filter written by hand straight into `similarity_search` — no LLM.

**WHY:** Path (c) is the ground truth: if the LLM-parsed filter (b) returns
the same bucket-b1 set as the hand-written filter, the self-query pipeline
provably extracted the constraint correctly. All three artifact sets live in
one `exp` for demo and gate to share.


In [5]:
def run_experiment() -> dict:
    texts = load_passages(PASSAGES_PATH, N_PASSAGES)
    docs = attach_metadata(texts)

    # --- Embed + index into an EPHEMERAL (in-memory) Chroma store ----------
    embedder = HuggingFaceEmbeddings(
        model_name=EMBED_MODEL_NAME, encode_kwargs={"normalize_embeddings": True}
    )
    vs = Chroma(embedding_function=embedder)  # no persist_directory, no collection
    t0 = time.perf_counter()
    vs.add_documents(docs)
    index_s = time.perf_counter() - t0
    n_indexed = len(vs.get()["ids"])

    # --- The query parser: Groq turns natural language into a filter --------
    load_dotenv(REPO_ROOT / ".env")  # GROQ_API_KEY lives at the repo root
    llm = ChatGroq(model=LLM_MODEL, temperature=LLM_TEMPERATURE)
    # (Gemini alternative: llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash",
    #  temperature=LLM_TEMPERATURE))

    fields = [
        AttributeInfo(
            name="bucket",
            description="the topic bucket label, one of b1, b2 or b3",
            type="string",
        ),
        AttributeInfo(
            name="source", description="the corpus source name", type="string"
        ),
    ]
    # ChromaTranslator passed explicitly: langchain-classic 1.0.8's auto
    # translator detection crashes on langchain-community 1.x (VERIFIED).
    retriever = SelfQueryRetriever.from_llm(
        llm=llm,
        vectorstore=vs,
        document_contents="financial FAQ passages from a Q&A corpus",
        metadata_field_info=fields,
        structured_query_translator=ChromaTranslator(),
        search_kwargs={"k": K},
    )

    # --- (a) Pure semantic query: no filter expected -----------------------
    t0 = time.perf_counter()
    parsed_plain = retriever.query_constructor.invoke({"query": QUERY_PLAIN})
    parse_plain_s = time.perf_counter() - t0
    t0 = time.perf_counter()
    res_plain = retriever.invoke(QUERY_PLAIN)
    plain_s = time.perf_counter() - t0

    # --- (b) Same sentence + NL constraint: a filter must be emitted --------
    t0 = time.perf_counter()
    parsed_filtered = retriever.query_constructor.invoke({"query": QUERY_FILTERED})
    parse_filtered_s = time.perf_counter() - t0

    # The gate's contract depends on the LLM actually emitting the filter.
    # If a rare parse failure slips through, retry ONLY this invoke (max 2);
    # the gate itself stays honest and reports whatever the last run returned.
    t0 = time.perf_counter()
    res_filtered, parse_attempts = _invoke_filtered(
        retriever, QUERY_FILTERED, FILTER_BUCKET
    )
    filtered_s = time.perf_counter() - t0

    # --- (c) The same filter written by hand, no LLM involved ---------------
    t0 = time.perf_counter()
    res_plain_filter = vs.similarity_search(
        QUERY_PLAIN, k=K, filter={"bucket": FILTER_BUCKET}
    )
    plain_filter_s = time.perf_counter() - t0

    return {
        "texts": texts,
        "docs": docs,
        "retriever": retriever,
        "n_indexed": n_indexed,
        "index_s": index_s,
        "dim": len(embedder.embed_query("probe")),
        "parsed_plain": parsed_plain,
        "parsed_filtered": parsed_filtered,
        "res_plain": res_plain,
        "res_filtered": res_filtered,
        "res_plain_filter": res_plain_filter,
        "parse_plain_s": parse_plain_s,
        "parse_filtered_s": parse_filtered_s,
        "plain_s": plain_s,
        "filtered_s": filtered_s,
        "plain_filter_s": plain_filter_s,
        "parse_attempts": parse_attempts,
    }


def _invoke_filtered(retriever: SelfQueryRetriever, query: str, bucket: str):
    """invoke() retried (max MAX_PARSE_ATTEMPTS) until the filter lands.

    Without the filter the store returns mixed buckets; with it, every doc
    comes from ``bucket``. That observable difference is the retry signal.
    Returns ``(result, attempts)`` so the demo can say whether a retry was
    needed.
    """
    for attempt in range(1, MAX_PARSE_ATTEMPTS + 1):
        res = retriever.invoke(query)
        if res and all(d.metadata.get("bucket") == bucket for d in res):
            return res, attempt
    return res, MAX_PARSE_ATTEMPTS


## 4 · Run — execute the experiment

**WHAT:** Calls `run_experiment()` — embedding, indexing, and a few Groq
parse calls (paths a and b) take a few seconds. The artifact dict is kept as
`exp`.

**WHY:** As in the earlier labs, demo and gate both read this single `exp`.
The gate's filter checks depend on the actual LLM-parsed filter, so the parse
calls must happen here, once.


In [6]:
exp = run_experiment()


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

## 5 · Demo — read the artifact

**WHAT:** `print_demo` prints the parsed structured queries (what the LLM
extracted from each question — filter or no filter), the retrieved top-K per
path with their bucket tags, and the parse/retrieval timings.

**WHY:** Compare path (a) to path (b): the same sentence, but (b)'s parsed
query carries `filter: bucket == b1` and the retrieved buckets collapse to
b1 — while (a) draws from at least two buckets. That contrast is self-query
working: the constraint moved from the sentence into the store's filter.


In [7]:
def print_demo(exp: dict) -> None:
    print("=" * 66)
    print("Lab 06 — Self-Query: an LLM writes the metadata filter")
    print(f"{EMBED_MODEL_NAME} | ephemeral Chroma | {LLM_MODEL} as query parser")
    print("=" * 66)

    print(f"\n[1] Corpus + synthetic metadata:")
    print(f"    {exp['n_indexed']} passages (first {N_PASSAGES} of 3200)")
    print(f"    metadata per doc: bucket in {BUCKETS} (round-robin) "
          f"+ source={SOURCE_NAME}")
    print(f"    e.g. [{exp['docs'][0].metadata}] {preview(exp['docs'][0].page_content)}")

    print(f"\n[2] Embed + index (ephemeral — gone at process exit):")
    print(f"    {exp['n_indexed']} passages embedded (dim {exp['dim']}) "
          f"and indexed in {exp['index_s']:.2f}s")

    print(f"\n[3] (a) Pure semantic query — no filter:")
    print(f'    query: "{QUERY_PLAIN}"')
    print(f"    parsed by LLM: {exp['parsed_plain']} ({exp['parse_plain_s']:.1f}s)")
    print(f"    returned: {len(exp['res_plain'])} docs, "
          f"buckets = {buckets_of(exp['res_plain'])}")
    print("      ^ filter=None -> store searched ALL buckets; see how the")
    print("        buckets above span more than one label.")

    print(f"\n[4] (b) Same sentence + natural-language constraint:")
    print(f'    query: "{QUERY_FILTERED}"')
    print(f"    parsed by LLM: {exp['parsed_filtered']} ({exp['parse_filtered_s']:.1f}s)")
    print(f"    returned: {len(exp['res_filtered'])} docs, "
          f"buckets = {buckets_of(exp['res_filtered'])}")
    print("      ^ the LLM emitted Comparison(eq, bucket, b1); the store")
    print("        applied it and every returned doc is from bucket b1.")
    if exp["parse_attempts"] > 1:
        print(f"      (filter parse needed {exp['parse_attempts']} attempts)")
    for rank, doc in enumerate(exp["res_filtered"], 1):
        print(f"      {rank}. [bucket {doc.metadata['bucket']}] {preview(doc.page_content)}")

    print(f"\n[5] (c) The same filter written by hand — no LLM:")
    print(f'    query: "{QUERY_PLAIN}", filter={{ "bucket": "{FILTER_BUCKET}" }}')
    print(f"    returned: {len(exp['res_plain_filter'])} docs, "
          f"buckets = {buckets_of(exp['res_plain_filter'])}")
    print("      ^ identical store behaviour — the only difference vs (b) is")
    print("        WHO wrote the filter: a human/API here, the LLM in (b).")

    print("\n[6] Takeaway")
    print("    The vector index cannot see constraints; metadata filters can,")
    print("    but someone must author them. Self-querying makes the LLM that")
    print("    author: one prompt maps 'from bucket b1' to a structured")
    print("    Comparison filter, and the store stays a plain filtered search.")
    print("    Cost: one extra LLM call per query — the trade for scoped,")
    print("    English-driven retrieval.")


In [8]:
print_demo(exp)


Lab 06 — Self-Query: an LLM writes the metadata filter
BAAI/bge-base-en-v1.5 | ephemeral Chroma | llama-3.3-70b-versatile as query parser

[1] Corpus + synthetic metadata:
    60 passages (first 60 of 3200)
    metadata per doc: bucket in ('b1', 'b2', 'b3') (round-robin) + source=rag-mini-wikipedia
    e.g. [{'bucket': 'b1', 'source': 'rag-mini-wikipedia'}] Uruguay (official full name in  ; pron.  , Eastern Republic of...

[2] Embed + index (ephemeral — gone at process exit):
    60 passages embedded (dim 768) and indexed in 0.89s

[3] (a) Pure semantic query — no filter:
    query: "Who founded Montevideo?"
    parsed by LLM: query='founded Montevideo' filter=None limit=None (0.4s)
    returned: 5 docs, buckets = ['b1', 'b1', 'b3', 'b3', 'b2']
      ^ filter=None -> store searched ALL buckets; see how the
        buckets above span more than one label.

[4] (b) Same sentence + natural-language constraint:
    query: "Who founded Montevideo? from bucket b1"
    parsed by LLM: query='fo

## 6 · Verification gate — the same checks the .py runs

**WHAT:** Runs the exact `verify_gate`: BGE dimension, exactly 60 indexed,
path (a) returning exactly K docs with >= 2 distinct buckets (unfiltered),
path (b) having the LLM emit a filter and returning only bucket-b1 docs
(0 violations), and path (c) returning exactly K bucket-b1 docs.

**WHY:** `python 06-self-query.py --verify` must print 8/8 PASS; this cell
proves the notebook reproduces the verified `.py` exactly. The zero-violation
check is the strong claim: every returned doc, not just the top-1, respects
the parsed filter.


In [9]:
def verify_gate(exp: dict) -> int:
    checks: list[tuple[str, bool]] = []

    # Store structure matches the config.
    checks.append(("embedding dimension is 768 (BGE base)", exp["dim"] == EMBED_DIM))
    checks.append((f"exactly {N_PASSAGES} passages indexed", exp["n_indexed"] == N_PASSAGES))

    # (a) Pure semantic query: top-K returned, visibly unfiltered (the store
    # was free to pick any bucket — at least two different buckets appear).
    buckets_a = buckets_of(exp["res_plain"])
    checks.append(("query (a) returns exactly K docs", len(exp["res_plain"]) == K))
    checks.append(
        ("query (a) is unfiltered: >= 2 distinct buckets in top-K",
         len(set(buckets_a)) >= 2)
    )

    # (b) The contract: the LLM emitted a filter AND the store honoured it.
    buckets_b = buckets_of(exp["res_filtered"])
    checks.append(
        ("query (b): LLM parsed a metadata filter",
         exp["parsed_filtered"].filter is not None)
    )
    checks.append(
        (f"query (b): >= 1 doc returned", len(exp["res_filtered"]) >= 1)
    )
    checks.append(
        (f"query (b): every doc is bucket {FILTER_BUCKET} (0 violations)",
         len(buckets_b) >= 1 and all(b == FILTER_BUCKET for b in buckets_b))
    )

    # (c) Hand-written filter: same constraint, same result shape.
    buckets_c = buckets_of(exp["res_plain_filter"])
    checks.append(
        (f"query (c): exactly K docs, all bucket {FILTER_BUCKET}",
         len(exp["res_plain_filter"]) == K
         and all(b == FILTER_BUCKET for b in buckets_c))
    )

    print("verification gate:")
    for label, ok in checks:
        print(f"  [{'PASS' if ok else 'FAIL'}] {label}")
    return 0 if all(ok for _, ok in checks) else 1


In [10]:
verify_gate(exp)


verification gate:
  [PASS] embedding dimension is 768 (BGE base)
  [PASS] exactly 60 passages indexed
  [PASS] query (a) returns exactly K docs
  [PASS] query (a) is unfiltered: >= 2 distinct buckets in top-K
  [PASS] query (b): LLM parsed a metadata filter
  [PASS] query (b): >= 1 doc returned
  [PASS] query (b): every doc is bucket b1 (0 violations)
  [PASS] query (c): exactly K docs, all bucket b1


0